## Actividad 3_20: Perros y gatos
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para enseñar a este software a diferenciar entre perros y gatos.

Para ello vamos a cargar los datos y etiquetarlos, a lanzar un Random Forest Classifier para establecer un punto de partida que debemos mejorar y después, vamos a tratar de solucionar el problema con una red neuronal convencional.
</div>

In [1]:
import tensorflow as tf
import gc
from tensorflow.keras import backend as K
import os

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Evitar que TensorFlow reserve toda la GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU lista")
print(tf.config.list_physical_devices("GPU"))
def reset_tf():
    K.clear_session()
    gc.collect()

GPU lista
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./PetImages/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [3]:
#2. IMPORTAMOS LOS DATOS:
for idx,folder in enumerate(folders):
    for file in listdir('./PetImages/'+folder):
        #Cargamos la imagen.
        #load_img sirve para cargar las imágenes en memoria. Tiene distintos parámetros para modificar como se cargan las imágenes.
        photo = load_img('./PetImages/'+folder+'/' + file, target_size=(128, 128)) 
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

/home/ciabd10/anaconda3/envs/tf3060/lib/python3.10/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


0
1


In [4]:
photos = asarray(photos)
labels = asarray(labels)

In [5]:
from sklearn.model_selection import train_test_split

# Datos originales
X = photos / 255.0
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
print(X_train.shape, y_train.shape)

(19998, 128, 128, 3) (19998,)


In [7]:
# Vamos a empezar con un RandomForest para comprobar que tal se clasifican las fotos
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, bootstrap=False)
forest_clf.fit(X_train, y_train)

y_pred = forest_clf.predict(X_test)

from sklearn.metrics import accuracy_score
print("Accuracy", accuracy_score(y_test, y_pred))

ValueError: Found array with dim 4. RandomForestClassifier expected <= 2.

### Haciendolo con red neuronal

In [8]:
print(X_train.shape)
print(y_train[:10])

(19998, 128, 128, 3)
[1. 1. 0. 1. 0. 0. 0. 1. 1. 0.]


In [9]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(128, 128, 3)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(128, activation="relu"))
model.add(keras.layers.Dense(64, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(2, activation="softmax"))

from tensorflow.keras import optimizers
sgd = optimizers.Adam(learning_rate=0.0005)

model.compile(loss="sparse_categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

2026-04-23 19:12:50.681285: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-23 19:12:50.681431: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-23 19:12:50.681512: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [10]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[early_stopping_cb], batch_size=8)

Epoch 1/100


2026-04-23 19:12:53.366019: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-23 19:12:53.417472: I external/local_xla/xla/service/service.cc:168] XLA service 0x629a9d1dd460 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-23 19:12:53.417488: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-04-23 19:12:53.421434: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-23 19:12:53.431656: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1776964373.490827  136154 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2250/2250 [==============================] - 7s 3ms/step - loss: 0.7800 - accuracy: 0.5377 - val_loss: 0.6870 - val_accuracy: 0.5535
Epoch 2/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6819 - accuracy: 0.5659 - val_loss: 0.6652 - val_accuracy: 0.5990
Epoch 3/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6824 - accuracy: 0.5598 - val_loss: 0.6734 - val_accuracy: 0.5695
Epoch 4/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6739 - accuracy: 0.5810 - val_loss: 0.6763 - val_accuracy: 0.5535
Epoch 5/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6723 - accuracy: 0.5762 - val_loss: 0.6888 - val_accuracy: 0.5520
Epoch 6/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6736 - accuracy: 0.5791 - val_loss: 0.6780 - val_accuracy: 0.5545
Epoch 7/100
2250/2250 [==============================] - 5s 2ms/step - loss: 0.6725 - accuracy: 0.5751 - val_loss: 0.6770 - val_accuracy: 0.57

In [11]:
model.evaluate(X_test, y_test)

157/157 [==============================] - 0s 2ms/step - loss: 0.6637 - accuracy: 0.6082


[0.6637458205223083, 0.6082000136375427]

In [12]:
del model
reset_tf()

## Ahora con red neuronal convolucional

In [13]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3), kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

# Capa convolucional adicional
model_cnn.add(keras.layers.Conv2D(64, (3, 3), activation="relu", kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(64, activation="relu"))
model_cnn.add(keras.layers.Dense(2, activation="softmax"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.001)
model_cnn.compile(optimizer=sgd_cnn, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [14]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20


2026-04-23 19:14:10.287450: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


563/563 [==============================] - 9s 11ms/step - loss: 0.5863 - accuracy: 0.6887 - val_loss: 0.5925 - val_accuracy: 0.6930
Epoch 2/20
563/563 [==============================] - 5s 9ms/step - loss: 0.4450 - accuracy: 0.7949 - val_loss: 0.4804 - val_accuracy: 0.7650
Epoch 3/20
563/563 [==============================] - 5s 9ms/step - loss: 0.3580 - accuracy: 0.8403 - val_loss: 0.5056 - val_accuracy: 0.7610
Epoch 4/20
563/563 [==============================] - 5s 10ms/step - loss: 0.2655 - accuracy: 0.8900 - val_loss: 0.5339 - val_accuracy: 0.7760
Epoch 5/20
563/563 [==============================] - 5s 10ms/step - loss: 0.1649 - accuracy: 0.9368 - val_loss: 0.6406 - val_accuracy: 0.7700
Epoch 6/20
563/563 [==============================] - 5s 9ms/step - loss: 0.0849 - accuracy: 0.9688 - val_loss: 0.8219 - val_accuracy: 0.7905
Epoch 7/20
563/563 [==============================] - 5s 9ms/step - loss: 0.0503 - accuracy: 0.9839 - val_loss: 0.8126 - val_accuracy: 0.7950
Epoch 8/20
563

In [15]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 5ms/step - loss: 0.4502 - accuracy: 0.7976


[0.4501631259918213, 0.7975999712944031]

In [16]:
tf.keras.backend.clear_session()

## Otra red convolucional

In [17]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(16, (5, 5), activation="relu", input_shape=(128, 128, 3), kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

# Capa convolucional adicional
model_cnn.add(keras.layers.Conv2D(32, (5, 5), activation="relu", kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(32, activation="relu"))
model_cnn.add(keras.layers.Dense(2, activation="softmax"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.001)
model_cnn.compile(optimizer=sgd_cnn, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [18]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20
563/563 [==============================] - 7s 9ms/step - loss: 0.6414 - accuracy: 0.6361 - val_loss: 0.6079 - val_accuracy: 0.6770
Epoch 2/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5118 - accuracy: 0.7489 - val_loss: 0.5208 - val_accuracy: 0.7465
Epoch 3/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4320 - accuracy: 0.7931 - val_loss: 0.5116 - val_accuracy: 0.7500
Epoch 4/20
563/563 [==============================] - 4s 7ms/step - loss: 0.3388 - accuracy: 0.8469 - val_loss: 0.5212 - val_accuracy: 0.7680
Epoch 5/20
563/563 [==============================] - 4s 7ms/step - loss: 0.2313 - accuracy: 0.9017 - val_loss: 0.6917 - val_accuracy: 0.7455
Epoch 6/20
563/563 [==============================] - 4s 7ms/step - loss: 0.1342 - accuracy: 0.9475 - val_loss: 0.7376 - val_accuracy: 0.7615
Epoch 7/20
563/563 [==============================] - 4s 7ms/step - loss: 0.0690 - accuracy: 0.9756 - val_loss: 0.9217 - val_accuracy: 0.7580
Epoch 

In [19]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 4ms/step - loss: 0.4973 - accuracy: 0.7762


[0.4973161220550537, 0.776199996471405]

In [20]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

del model_cnn
reset_tf()

In [21]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

model_cnn = keras.Sequential([
    layers.Conv2D(16, (5,5), activation="relu", input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (5,5), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(48, activation="relu",
                 kernel_regularizer=regularizers.l2(0.0001)),
    layers.Dropout(0.3),

    layers.Dense(2, activation="softmax")
])

model_cnn.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20
563/563 [==============================] - 6s 8ms/step - loss: 0.6960 - accuracy: 0.5287 - val_loss: 0.6790 - val_accuracy: 0.5900
Epoch 2/20
563/563 [==============================] - 4s 7ms/step - loss: 0.6437 - accuracy: 0.6486 - val_loss: 0.6535 - val_accuracy: 0.6515
Epoch 3/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5944 - accuracy: 0.6993 - val_loss: 0.5909 - val_accuracy: 0.7215
Epoch 4/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5455 - accuracy: 0.7433 - val_loss: 0.5995 - val_accuracy: 0.7180
Epoch 5/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5024 - accuracy: 0.7750 - val_loss: 0.6267 - val_accuracy: 0.7190
Epoch 6/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4592 - accuracy: 0.8044 - val_loss: 0.6327 - val_accuracy: 0.7205
Epoch 7/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4174 - accuracy: 0.8297 - val_loss: 0.6286 - val_accuracy: 0.7350
Epoch 

In [23]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 3ms/step - loss: 0.5842 - accuracy: 0.7160


[0.5842381119728088, 0.7160000205039978]